# 05. 演習2 — ハイパーパラメータ改善（Sweep Job）

**対応するテキスト**: [docs/08_演習2_ハイパーパラメータ改善.md](../docs/08_演習2_ハイパーパラメータ改善.md)

2 本の Sweep を実行します。

| # | 目的 | 探索空間 |
|---|---|---|
| **A** | **再現性の評価**（同一設定でどれだけばらつくか） | `seed` のみ |
| **B** | ハイパーパラメーター探索 | `learning_rate` × `gamma` × `seed` |

> ⚠ **A を先に実行してください。**
> 「同一設定でのばらつき」を知らないと、B の結果を解釈できません。
> 強化学習では、**単なる乱数の揺れを「改善」と誤認するのが最も多い失敗**です。

In [ ]:
from azure.ai.ml import MLClient, command
from azure.ai.ml.sweep import Choice
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<SUBSCRIPTION_ID>"
RESOURCE_GROUP = "<RESOURCE_GROUP>"
WORKSPACE_NAME = "<AML_WORKSPACE_NAME>"

COMPUTE_NAME = "cpu-cluster"
ENV_REF = "rl-panda-gym-env@latest"
EXPERIMENT = "rl-hparam-sweep"

#  ⚠ クラスターの max_instances とクォータを超えないこと
MAX_CONCURRENT = 3

TAGS = {
    "project": "rl-workshop",
    "owner": "<your-alias>",
    "delete-after": "<YYYY-MM-DD>",
    "phase": "hparam",
}

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
ws = ml_client.workspaces.get(WORKSPACE_NAME)
print("接続しました:", ws.name)

## 1. Sweep 用の trial ジョブを定義する

Sweep では、**探索したい値を `inputs` として宣言し、コマンド内で `${{inputs.名前}}` として参照**します。

> 出典: [Hyperparameter tuning a model (v2) - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-tune-hyperparameters?view=azureml-api-2)

> ⚠ **`primary_metric` は学習スクリプトが記録するメトリック名と完全一致していなければなりません。**
> 本テキストではメトリック名を `final_success_rate` のように平坦にしています（`/` を含めていません）。

In [ ]:
TRIAL_COMMAND = (
    "python train_rl.py"
    " --env-id PandaPickAndPlace-v3"
    " --use-her 1"
    " --algo sac"
    " --reward-mode sparse"
    " --total-timesteps 50000"
    " --batch-size 256"
    " --buffer-size 200000"
    " --learning-starts 1000"
    " --eval-freq 5000"
    " --n-eval-episodes 30"
    " --final-eval-episodes 100"
    " --reward-fn-version v1"
    " --seed ${{inputs.seed}}"
    " --learning-rate ${{inputs.learning_rate}}"
    " --gamma ${{inputs.gamma}}"
)

trial_job = command(
    code="../src",
    command=TRIAL_COMMAND,
    inputs=dict(seed=0, learning_rate=3e-4, gamma=0.99),   # 既定値（ベースライン）
    environment=ENV_REF,
    compute=COMPUTE_NAME,
    tags=TAGS,
)
print(TRIAL_COMMAND)

## 2. 演習 A: シードのみを振る（再現性の評価）

**ハイパーパラメーターは一切変えず、`seed` だけを 3 通り試します。**

> **これが本章で最も重要な実験です。**
> ここで得られる「ばらつきの幅」が、以降のすべての比較の**判定基準**になります。

In [ ]:
sweep_a = trial_job(seed=Choice(values=[0, 1, 2])).sweep(
    compute=COMPUTE_NAME,
    sampling_algorithm="grid",
    primary_metric="final_success_rate",   # ← train_rl.py が記録する名前と完全一致
    goal="Maximize",
    max_total_trials=3,
    max_concurrent_trials=MAX_CONCURRENT,
)
#  実験名と表示名は Sweep ノードに設定する。
#  （.sweep() の引数には experiment_name / display_name が無いため、生成後に代入する）
sweep_a.experiment_name = EXPERIMENT
sweep_a.display_name = "sweepA_seed-only_reproducibility"

sweep_a_job = ml_client.jobs.create_or_update(sweep_a)
print("Sweep A:", sweep_a_job.name)
print("studio :", sweep_a_job.studio_url)
print("実験名 :", getattr(sweep_a_job, "experiment_name", "(取得できません)"))

## 3. 演習 B: ハイパーパラメーター探索

`learning_rate` × `gamma` × `seed` の格子探索です。

> ⚠ **コストの上限を必ず設定してください。**
> `max_total_trials` を指定しないと、探索空間の全組み合わせが実行されます。
>
> 概算コスト ≒ (1 trial の学習時間) × (試行数) × (VM の時間単価)
> → 単価は [Azure Machine Learning 価格](https://azure.microsoft.com/pricing/details/machine-learning/) で確認してください。

In [ ]:
#  探索空間: 3 学習率 × 2 割引率 × 2 シード = 12 通り
#  ただし max_total_trials で 12 に上限を設ける
sweep_b = trial_job(
    learning_rate=Choice(values=[1e-4, 3e-4, 1e-3]),
    gamma=Choice(values=[0.95, 0.99]),
    seed=Choice(values=[0, 1]),
).sweep(
    compute=COMPUTE_NAME,
    sampling_algorithm="grid",
    primary_metric="final_success_rate",
    goal="Maximize",
    max_total_trials=12,           # ← コストの上限
    max_concurrent_trials=MAX_CONCURRENT,
    timeout=6 * 60 * 60,           # ← 全体の時間上限（秒）
)
sweep_b.experiment_name = EXPERIMENT
sweep_b.display_name = "sweepB_lr-gamma-seed_grid"

sweep_b_job = ml_client.jobs.create_or_update(sweep_b)
print("Sweep B:", sweep_b_job.name)
print("studio :", sweep_b_job.studio_url)

### 早期終了ポリシーについて（本演習では使いません）

> ⚠⚠ **強化学習では早期終了に注意が必要です。**
> **強化学習は「長い平坦期のあとに急に立ち上がる」ことが珍しくありません。**
> 素直に早期終了を使うと、**あとで伸びるはずの設定を序盤で切ってしまいます。**
>
> 使う場合は `delay_evaluation` を必ず設定してください。
>
> ```python
> from azure.ai.ml.sweep import BanditPolicy
> early_termination = BanditPolicy(
>     slack_factor=0.2, evaluation_interval=1, delay_evaluation=5
> )
> ```
>
> 出典: [Specify early termination policy - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-tune-hyperparameters?view=azureml-api-2#early-termination)

In [ ]:
for name, job in [("Sweep A", sweep_a_job), ("Sweep B", sweep_b_job)]:
    print(f"{name}: {ml_client.jobs.get(job.name).status}")

## 4. 結果を集計する

Sweep の子ジョブ（trial）は **親ジョブと同じ実験に記録されます**。
`mlflow.search_runs()` でまとめて取得します。

> 出典: [Query & compare experiments and runs with MLflow - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-track-experiments-mlflow?view=azureml-api-2)

In [ ]:
import mlflow
import pandas as pd

try:
    tracking_uri = ml_client.workspaces.get(WORKSPACE_NAME).mlflow_tracking_uri
except AttributeError:
    tracking_uri = (
        f"azureml://{ws.location}.api.azureml.ms/mlflow/v1.0"
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
    )
mlflow.set_tracking_uri(tracking_uri)

runs = mlflow.search_runs(experiment_names=[EXPERIMENT], output_format="pandas")

COLS = {
    "params.seed": "seed",
    "params.learning_rate": "learning_rate",
    "params.gamma": "gamma",
    "metrics.final_success_rate": "success_rate",
    "metrics.final_success_rate_stderr": "stderr",
    "metrics.final_mean_reward": "mean_reward",
    "metrics.final_std_reward": "std_reward",
    "metrics.train_minutes": "train_minutes",
}
available = {k: v for k, v in COLS.items() if k in runs.columns}
df = runs[list(available)].rename(columns=available).dropna(subset=["success_rate"])
for c in ["seed", "learning_rate", "gamma"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.sort_values("success_rate", ascending=False)
df.to_csv("sweep_results.csv", index=False, encoding="utf-8-sig")
df

## 5. 【最重要】シード間のばらつきと比べる

> **強化学習で最も多い誤りは、「単なる乱数の揺れ」を「改善」と誤認することです。**
>
> **判定基準**: パラメーター間の差が、**同一設定のシード間のばらつきより明確に大きい**か。

In [ ]:
# --- 同一設定でのシード間ばらつき（ベースライン設定に絞る） ---
base = df[(df.get("learning_rate") == 3e-4) & (df.get("gamma") == 0.99)]
if len(base) >= 2:
    seed_spread = base["success_rate"].std()
    print(f"ベースライン設定のシード間 標準偏差 : {seed_spread:.4f}")
    print(f"  成功率の値: {sorted(base['success_rate'].round(3).tolist())}")
else:
    seed_spread = float("nan")
    print("[WARN] ベースライン設定の trial が 2 本未満です。Sweep A の完了を待ってください。")

# --- 学習率ごとの平均と標準偏差 ---
if "learning_rate" in df.columns:
    grouped = df.groupby("learning_rate")["success_rate"].agg(["mean", "std", "count"])
    print("\n学習率ごとの成功率:")
    print(grouped)

    if len(grouped) >= 2 and seed_spread == seed_spread:  # NaN でない
        span = grouped["mean"].max() - grouped["mean"].min()
        print(f"\n学習率間の平均の差 : {span:.4f}")
        print(f"シード間ばらつき   : {seed_spread:.4f}")
        if span > 2 * seed_spread:
            print("→ 学習率の効果は、シードのばらつきより明確に大きい（効果ありと言える）")
        else:
            print("→ 差はシードのばらつきに埋もれている（効果ありとは言えない）")

## 6. 可視化

**studio の［平行座標プロット］も必ず見てください。**
Sweep Job の詳細ページから、メトリック チャート・平行座標プロット・2 次元散布図が利用できます。

> 出典: [Visualize hyperparameter tuning jobs - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-tune-hyperparameters?view=azureml-api-2#visualize-hyperparameter-tuning-jobs)

In [ ]:
import matplotlib.pyplot as plt

if {"learning_rate", "gamma", "success_rate"}.issubset(df.columns) and len(df) > 0:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for gamma_value, sub in df.groupby("gamma"):
        agg = sub.groupby("learning_rate")["success_rate"].agg(["mean", "std"])
        ax.errorbar(agg.index, agg["mean"], yerr=agg["std"].fillna(0),
                    marker="o", capsize=4, label=f"gamma={gamma_value}")
    ax.set_xscale("log")
    ax.set_xlabel("learning_rate (log scale)")
    ax.set_ylabel("final_success_rate")
    ax.set_title("学習率と成功率（エラーバー = シード間の標準偏差）")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("データが足りません。Sweep の完了を待ってください。")

## 7. ✅ チェックリスト

- [ ] **Sweep A（シードのみ）を先に実行した**
- [ ] 同一設定でのばらつきの大きさを把握した
- [ ] Sweep B（ハイパーパラメーター探索）を実行した
- [ ] **`max_total_trials` と `timeout` を設定した**（コスト上限）
- [ ] studio の **平行座標プロット**を確認した
- [ ] **差がシード間のばらつきより大きいかを判定した**（5 節）
- [ ] `sweep_results.csv` を保存した
- [ ] **「効果を確認できなかった」条件も記録した**

→ 次は [docs/09_評価・コスト・後片付け.md](../docs/09_評価・コスト・後片付け.md) と [06_compare_and_cleanup.ipynb](06_compare_and_cleanup.ipynb) へ。